# `3.repeat_until_success` — heralded $|1\rangle$ preparation

This notebook is the MLIR-stack counterpart of
[examples/3.repeat_until_success.ipynb](../../examples/3.repeat_until_success.ipynb).

It also corresponds to the first end-to-end milestone of the MLIR
refactor — already captured in [`prepare_one.ipynb`](prepare_one.ipynb)
— but using the same numbering as the legacy `examples/` directory so
the two trees line up.

The pattern: prepare a heralded $|1\rangle$ by entangling the data
qubit with an ancilla, measuring the ancilla, and **retrying** if the
herald didn't fire. Recursion through a `switch` over a selector is the
runtime's way of expressing "ask the host whether to run another round".

In [ ]:
%load_ext qstack_mlir.jupyter

## 1. The program

`prepare_one(q)` recursively reprepares `q` until the ancilla
measurement returns `1`. The `extern selector` is the host-language
hook the runtime calls each iteration to ask "another round?".

In [ ]:
%%qasm prep
QSTACKQASM 0.1;
include "qstack/cliffords.inc";

extern selector repeat_until_one(bit) -> int;

def prepare_one(qubit q) {
  qreg ancilla[1];
  bit m;
  h q;
  cx q, ancilla[0];
  measure ancilla[0] -> m;
  switch (repeat_until_one(m)) {
    case 0: { }                 // done
    case 1: { prepare_one q; }  // retry
  }
}

qreg q[1];
creg c[1];
prepare_one q[0];
measure q[0] -> c[0];

## 2. Inspect the lowered IR

`switch` becomes a `qstack.select` with a two-entry continuation menu
(`__case_0_*`, `__case_1_*`); the latter recursively `func.call`s
`@prepare_one`.

In [ ]:
print(prep)

## 3. Run

Register the selector, hand the module to a `Machine`, run 1000 shots.
Every shot must return `1` — that's the whole point of the protocol.

In [ ]:
from qstack_mlir.runtime import CallbackRegistry, Machine

reg = CallbackRegistry()

@reg.selector("repeat_until_one")
def _pick(*, b0):
    # ancilla=1 -> done (case 0); ancilla=0 -> retry (case 1)
    return "0" if b0 == 1 else "1"

machine = Machine(prep, num_qubits=4, registry=reg)
machine.shots("main", 1000).plot_histogram()